# Chapter 3: Building Blocks -- A Mini RAG Pipeline from Scratch

**From *AI Enterprise Architecture***

This notebook builds a Retrieval-Augmented Generation (RAG) pipeline step by step,
using only OpenAI's API and NumPy -- no external vector database required.

You will see each component in action:

1. **Embed** documents into vector representations
2. **Store** vectors in a simple in-memory store
3. **Retrieve** the most relevant documents for a query
4. **Generate** a grounded answer with citations

**Key insight:** RAG grounds AI in your organization's data without requiring
model fine-tuning -- and understanding its building blocks is essential for
designing reliable AI architectures.

## Setup

In [ ]:
!pip install -q openai numpy tabulate

In [ ]:
import os
import numpy as np
from openai import OpenAI
from tabulate import tabulate

# Set your OpenAI API key as an environment variable before running.
# In Colab: use the Secrets panel (key icon in the left sidebar),
# then uncomment the two lines below:
# from google.colab import userdata
# os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise ValueError(
        "OPENAI_API_KEY not found. Set it as an environment variable "
        "or add it via Colab Secrets."
    )

client = OpenAI(api_key=api_key)
print("OpenAI client ready.")

## Step 1: Create a Knowledge Base

We define a small set of documents representing a fictional company's internal
policies. In a real system these would come from a document store, wiki, or
database -- but the RAG pattern is the same regardless of source.

In [ ]:
DOCUMENTS = [
    {
        "id": "DOC-001",
        "title": "Remote Work Policy",
        "content": (
            "Employees may work remotely up to 3 days per week. Remote work must be "
            "approved by the direct manager. Employees must be available during core "
            "hours (10 AM - 3 PM local time). A stable internet connection is required. "
            "Remote workers must use the company VPN for all work-related activities."
        ),
    },
    {
        "id": "DOC-002",
        "title": "Expense Reimbursement",
        "content": (
            "Business expenses must be submitted within 30 days of the transaction. "
            "Receipts are required for all expenses over $25. Travel meals are capped "
            "at $75 per day. International travel requires pre-approval from a VP. "
            "Software subscriptions must go through IT procurement."
        ),
    },
    {
        "id": "DOC-003",
        "title": "Data Classification Policy",
        "content": (
            "All company data must be classified into one of four levels: Public, "
            "Internal, Confidential, and Restricted. Customer PII is always Restricted. "
            "Financial reports are Confidential until published. Source code is Internal. "
            "Marketing materials are Public after approval."
        ),
    },
    {
        "id": "DOC-004",
        "title": "Incident Response Procedure",
        "content": (
            "Security incidents must be reported to the SOC within 1 hour of discovery. "
            "Severity levels: P1 (data breach), P2 (service outage), P3 (vulnerability), "
            "P4 (policy violation). P1 incidents require immediate executive notification. "
            "All incidents must have a post-mortem within 5 business days."
        ),
    },
    {
        "id": "DOC-005",
        "title": "AI Usage Guidelines",
        "content": (
            "Employees may use approved AI tools for productivity. Confidential or "
            "Restricted data must never be entered into external AI services. All AI-generated "
            "content must be reviewed by a human before external publication. AI tools "
            "used in production systems must be registered in the AI inventory and "
            "undergo a risk assessment."
        ),
    },
    {
        "id": "DOC-006",
        "title": "Annual Leave Policy",
        "content": (
            "Full-time employees receive 20 days of paid annual leave per year. Leave "
            "must be requested at least 2 weeks in advance for periods longer than 3 days. "
            "Unused leave can be carried over up to 5 days into the next calendar year. "
            "Leave during blackout periods (year-end close) requires director approval."
        ),
    },
]

print(f"Knowledge base: {len(DOCUMENTS)} documents")
for doc in DOCUMENTS:
    print(f"  {doc['id']}: {doc['title']}")

## Step 2: Generate Embeddings

An **embedding** is a vector (list of numbers) that captures the semantic meaning
of a piece of text. Texts with similar meanings will have similar vectors.

We use OpenAI's `text-embedding-3-small` model, which produces 1536-dimensional vectors.

In [ ]:
EMBEDDING_MODEL = "text-embedding-3-small"


def get_embeddings(texts: list[str]) -> list[list[float]]:
    """Get embeddings for a list of texts using OpenAI's API."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=texts,
    )
    return [item.embedding for item in response.data]


# Embed all documents (we embed the title + content together for richer context)
doc_texts = [f"{doc['title']}: {doc['content']}" for doc in DOCUMENTS]
doc_embeddings = get_embeddings(doc_texts)

# Convert to numpy array for efficient computation
doc_matrix = np.array(doc_embeddings)

print(f"Embedded {len(doc_embeddings)} documents.")
print(f"Embedding dimension: {len(doc_embeddings[0])}")
print(f"Matrix shape: {doc_matrix.shape}")

## Step 3: Build an In-Memory Vector Store

In production you would use a dedicated vector database (Pinecone, Weaviate,
pgvector, etc.). Here we build a minimal version using NumPy to show
that the core operation is simply **cosine similarity** between vectors.

In [ ]:
class SimpleVectorStore:
    """A minimal in-memory vector store using cosine similarity."""

    def __init__(self, documents: list[dict], embeddings: np.ndarray):
        self.documents = documents
        # Normalize vectors once for fast cosine similarity later
        norms = np.linalg.norm(embeddings, axis=1, keepdims=True)
        self.normalized = embeddings / norms

    def search(self, query_embedding: list[float], top_k: int = 3) -> list[dict]:
        """Find the top-k most similar documents to the query."""
        query_vec = np.array(query_embedding)
        query_norm = query_vec / np.linalg.norm(query_vec)

        # Cosine similarity = dot product of normalized vectors
        similarities = self.normalized @ query_norm

        # Get top-k indices (sorted descending)
        top_indices = np.argsort(similarities)[::-1][:top_k]

        results = []
        for idx in top_indices:
            results.append({
                "document": self.documents[idx],
                "similarity": float(similarities[idx]),
            })
        return results


# Initialize the store
vector_store = SimpleVectorStore(DOCUMENTS, doc_matrix)
print("Vector store initialized with", len(DOCUMENTS), "documents.")

## Step 4: Query -- Retrieve Relevant Documents

To answer a question, we first **embed the query** using the same model, then
find the most similar documents in our store.

In [ ]:
def retrieve(query: str, top_k: int = 3) -> list[dict]:
    """Embed a query and retrieve the top-k most relevant documents."""
    query_embedding = get_embeddings([query])[0]
    return vector_store.search(query_embedding, top_k=top_k)


# Test retrieval with a sample question
test_query = "Can I work from home on Fridays?"
retrieved = retrieve(test_query)

print(f"Query: \"{test_query}\"\n")
print("Retrieved documents (ranked by relevance):\n")
for i, r in enumerate(retrieved, 1):
    doc = r["document"]
    print(f"  {i}. [{doc['id']}] {doc['title']}  (similarity: {r['similarity']:.4f})")
    print(f"     {doc['content'][:100]}...")
    print()

## Step 5: Build a Prompt with Retrieved Context

The key to RAG is injecting retrieved documents into the LLM's prompt.
This **grounds** the model's response in your actual data rather than
relying on its training knowledge.

In [ ]:
def build_rag_prompt(query: str, retrieved_docs: list[dict]) -> str:
    """Build a prompt that includes retrieved context and asks the LLM to answer."""
    context_parts = []
    for r in retrieved_docs:
        doc = r["document"]
        context_parts.append(
            f"[{doc['id']}] {doc['title']}:\n{doc['content']}"
        )

    context_block = "\n\n".join(context_parts)

    prompt = f"""You are a helpful company policy assistant. Answer the employee's question
based ONLY on the provided policy documents. If the answer is not in the documents,
say so. Always cite the document ID (e.g., [DOC-001]) when referencing a policy.

--- POLICY DOCUMENTS ---
{context_block}
--- END DOCUMENTS ---

Employee question: {query}

Answer:"""
    return prompt


# Preview the prompt
sample_prompt = build_rag_prompt(test_query, retrieved)
print("=== RAG PROMPT ===")
print(sample_prompt)
print(f"\n(Prompt length: {len(sample_prompt)} characters)")

## Step 6: Generate a Grounded Answer

Now we send the prompt (with context) to the LLM and get a grounded answer.

In [ ]:
def generate_answer(prompt: str) -> str:
    """Send the RAG prompt to OpenAI and return the generated answer."""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        temperature=0.2,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )
    return response.choices[0].message.content.strip()


answer = generate_answer(sample_prompt)

print(f"Question: {test_query}\n")
print(f"Answer:\n{answer}")

## The Full Pipeline: End-to-End

Let's wrap everything into a single function and run several questions through
the complete RAG pipeline.

In [ ]:
def ask(question: str, top_k: int = 3, verbose: bool = True) -> str:
    """Full RAG pipeline: retrieve -> build prompt -> generate answer."""
    # 1. Retrieve
    retrieved_docs = retrieve(question, top_k=top_k)

    if verbose:
        print(f"Retrieved {len(retrieved_docs)} documents:")
        for r in retrieved_docs:
            doc = r["document"]
            print(f"  - [{doc['id']}] {doc['title']} (sim: {r['similarity']:.3f})")
        print()

    # 2. Build prompt
    prompt = build_rag_prompt(question, retrieved_docs)

    # 3. Generate
    answer = generate_answer(prompt)
    return answer


print("Pipeline function ready.")

In [ ]:
# Run several questions through the full pipeline

QUESTIONS = [
    "Can I use ChatGPT for work projects?",
    "What's the maximum I can spend on dinner during a business trip?",
    "Someone found a security vulnerability -- what do I do?",
    "How many vacation days do I get per year?",
    "Is source code considered confidential?",
]

for q in QUESTIONS:
    print("=" * 70)
    print(f"QUESTION: {q}\n")
    answer = ask(q)
    print(f"ANSWER:\n{answer}")
    print()

## Bonus: What Happens Without RAG?

To demonstrate why retrieval matters, let's ask the same question with and
without the company documents. Without RAG, the model will either hallucinate
a policy or give a generic answer.

In [ ]:
comparison_question = "What's our company's policy on carrying over unused vacation days?"

# --- With RAG ---
print("WITH RAG (grounded in company documents):")
print("-" * 50)
rag_answer = ask(comparison_question, verbose=False)
print(rag_answer)

print()

# --- Without RAG ---
print("WITHOUT RAG (model relies on general knowledge):")
print("-" * 50)
no_rag_response = client.chat.completions.create(
    model="gpt-4o-mini",
    temperature=0.2,
    max_tokens=300,
    messages=[{
        "role": "user",
        "content": (
            "You are a company policy assistant. "
            f"Answer this question: {comparison_question}"
        ),
    }],
)
print(no_rag_response.choices[0].message.content.strip())

## Key Takeaways for Enterprise Architects

1. **RAG is a pattern, not a product.** The building blocks (embeddings, vector
   search, prompt construction, generation) can be assembled from different
   vendors and technologies.

2. **Retrieval quality determines answer quality.** If the wrong documents are
   retrieved, the generated answer will be wrong -- no matter how capable the LLM.
   Invest in chunking strategies, embedding models, and retrieval tuning.

3. **No model training required.** RAG lets you ground AI in your organization's
   data without fine-tuning. This dramatically reduces the barrier to entry.

4. **Citations enable trust.** By including document IDs in the prompt and
   instructing the model to cite them, users can verify answers against sources.

5. **Architecture decisions matter.** Chunking strategy, embedding model choice,
   number of retrieved documents, and prompt template design all affect output
   quality -- these are architectural decisions, not just implementation details.